# Positional Encoding


Use this notebook after the positional encoding note. The goal is to show exactly where position information enters attention.

You will:
- see that plain self-attention is permutation-equivariant
- add absolute and relative position information
- inspect the intuition behind rotary position embeddings


In [1]:
import math
from pathlib import Path

import torch
import torch.nn as nn

torch.manual_seed(0)

LECTURE_NOTE_TITLE = 'Positional Encoding'

def self_attention(x: torch.Tensor, bias: torch.Tensor | None = None) -> tuple[torch.Tensor, torch.Tensor]:
    scores = x @ x.T / math.sqrt(x.size(-1))
    if bias is not None:
        scores = scores + bias
    probs = torch.softmax(scores, dim=-1)
    return probs @ x, probs

def sinusoidal_positions(max_len: int, d_model: int) -> torch.Tensor:
    position = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
    pe = torch.zeros(max_len, d_model)
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe

print(f'Lecture note: {LECTURE_NOTE_TITLE}')


Lecture note: Positional Encoding


## Self-attention is permutation-equivariant without position


In [2]:
x = torch.tensor(
    [
        [1.0, 0.0, 1.0, 0.0],
        [0.0, 1.0, 0.0, 1.0],
        [1.0, 1.0, 0.0, 0.0],
        [0.0, 0.0, 1.0, 1.0],
    ]
)
perm = torch.tensor([2, 0, 3, 1])

out, _ = self_attention(x)
out_perm, _ = self_attention(x[perm])
restored = torch.empty_like(out_perm)
restored[perm] = out_perm

print('permutation-equivariant:', torch.allclose(out, restored, atol=1e-6))
print('original output:')
print(out)
print('permuted-then-restored output:')
print(restored)


permutation-equivariant: True
original output:
tensor([[0.6225, 0.3775, 0.6225, 0.3775],
        [0.3775, 0.6225, 0.3775, 0.6225],
        [0.6225, 0.6225, 0.3775, 0.3775],
        [0.3775, 0.3775, 0.6225, 0.6225]])
permuted-then-restored output:
tensor([[0.6225, 0.3775, 0.6225, 0.3775],
        [0.3775, 0.6225, 0.3775, 0.6225],
        [0.6225, 0.6225, 0.3775, 0.3775],
        [0.3775, 0.3775, 0.6225, 0.6225]])


## Sinusoidal absolute positions


In [3]:
pe = sinusoidal_positions(max_len=6, d_model=8)
print(pe)


tensor([[ 0.0000e+00,  1.0000e+00,  0.0000e+00,  1.0000e+00,  0.0000e+00,
          1.0000e+00,  0.0000e+00,  1.0000e+00],
        [ 8.4147e-01,  5.4030e-01,  9.9833e-02,  9.9500e-01,  9.9998e-03,
          9.9995e-01,  1.0000e-03,  1.0000e+00],
        [ 9.0930e-01, -4.1615e-01,  1.9867e-01,  9.8007e-01,  1.9999e-02,
          9.9980e-01,  2.0000e-03,  1.0000e+00],
        [ 1.4112e-01, -9.8999e-01,  2.9552e-01,  9.5534e-01,  2.9995e-02,
          9.9955e-01,  3.0000e-03,  1.0000e+00],
        [-7.5680e-01, -6.5364e-01,  3.8942e-01,  9.2106e-01,  3.9989e-02,
          9.9920e-01,  4.0000e-03,  9.9999e-01],
        [-9.5892e-01,  2.8366e-01,  4.7943e-01,  8.7758e-01,  4.9979e-02,
          9.9875e-01,  5.0000e-03,  9.9999e-01]])


In [4]:
repeated_token = torch.tensor([[2.0, -1.0, 0.5, 0.0, 2.0, -1.0, 0.5, 0.0]]).repeat(4, 1)
with_pos = repeated_token + sinusoidal_positions(max_len=4, d_model=8)

print('same token vector at every position:')
print(repeated_token)
print('after adding position vectors:')
print(with_pos)


same token vector at every position:
tensor([[ 2.0000, -1.0000,  0.5000,  0.0000,  2.0000, -1.0000,  0.5000,  0.0000],
        [ 2.0000, -1.0000,  0.5000,  0.0000,  2.0000, -1.0000,  0.5000,  0.0000],
        [ 2.0000, -1.0000,  0.5000,  0.0000,  2.0000, -1.0000,  0.5000,  0.0000],
        [ 2.0000, -1.0000,  0.5000,  0.0000,  2.0000, -1.0000,  0.5000,  0.0000]])
after adding position vectors:
tensor([[ 2.0000e+00,  0.0000e+00,  5.0000e-01,  1.0000e+00,  2.0000e+00,
          0.0000e+00,  5.0000e-01,  1.0000e+00],
        [ 2.8415e+00, -4.5970e-01,  5.9983e-01,  9.9500e-01,  2.0100e+00,
         -5.0008e-05,  5.0100e-01,  1.0000e+00],
        [ 2.9093e+00, -1.4161e+00,  6.9867e-01,  9.8007e-01,  2.0200e+00,
         -1.9997e-04,  5.0200e-01,  1.0000e+00],
        [ 2.1411e+00, -1.9900e+00,  7.9552e-01,  9.5534e-01,  2.0300e+00,
         -4.4996e-04,  5.0300e-01,  1.0000e+00]])


In [5]:
fixed_positions = sinusoidal_positions(max_len=4, d_model=4)
out_with_pos, _ = self_attention(x + fixed_positions)
out_perm_with_fixed_pos, _ = self_attention(x[perm] + fixed_positions)
restored = torch.empty_like(out_perm_with_fixed_pos)
restored[perm] = out_perm_with_fixed_pos

print('still permutation-equivariant after fixed positions?:', torch.allclose(out_with_pos, restored, atol=1e-6))


still permutation-equivariant after fixed positions?: False


## Learned absolute positional embeddings


In [6]:
vocab_size = 5
d_model = 6
seq_len = 4

token_embed = nn.Embedding(vocab_size, d_model)
pos_embed = nn.Embedding(seq_len, d_model)

token_ids = torch.tensor([1, 2, 1, 3])
position_ids = torch.arange(seq_len)
combined = token_embed(token_ids) + pos_embed(position_ids)

print('token contribution:')
print(token_embed(token_ids))
print('position contribution:')
print(pos_embed(position_ids))
print('sum passed to the model:')
print(combined)
print('learned position table params:', pos_embed.weight.numel())


token contribution:
tensor([[-0.3160, -2.1152,  0.3223, -1.2633,  0.3500,  0.3081],
        [ 0.1198,  1.2377, -0.1435, -0.1116, -0.6136,  0.0316],
        [-0.3160, -2.1152,  0.3223, -1.2633,  0.3500,  0.3081],
        [-0.4927,  0.2484,  0.4397,  0.1124, -0.8411, -2.3160]],
       grad_fn=<EmbeddingBackward0>)
position contribution:
tensor([[ 0.9383,  0.4889, -0.6731,  0.8728,  1.0554,  0.1778],
        [-0.2303, -0.3918, -0.4731,  0.3356,  1.5091,  2.0820],
        [ 1.7067,  2.3804, -1.1256, -0.3170, -0.1407,  0.8058],
        [ 0.3276, -0.7607, -1.5991,  0.0185, -0.7504,  0.1854]],
       grad_fn=<EmbeddingBackward0>)
sum passed to the model:
tensor([[ 0.6222, -1.6263, -0.3508, -0.3905,  1.4053,  0.4860],
        [-0.1105,  0.8459, -0.6166,  0.2239,  0.8955,  2.1135],
        [ 1.3907,  0.2651, -0.8033, -1.5803,  0.2093,  1.1139],
        [-0.1651, -0.5123, -1.1594,  0.1309, -1.5915, -2.1306]],
       grad_fn=<AddBackward0>)
learned position table params: 24


## Relative distance bias


In [7]:
seq_len = 5
distance = torch.arange(seq_len).unsqueeze(1) - torch.arange(seq_len).unsqueeze(0)
relative_bias = -distance.abs().float()

_, probs_without_bias = self_attention(torch.eye(seq_len))
_, probs_with_bias = self_attention(torch.eye(seq_len), bias=relative_bias)

print('relative bias matrix:')
print(relative_bias)
print('attention without bias:')
print(probs_without_bias)
print('attention with relative bias:')
print(probs_with_bias)


relative bias matrix:
tensor([[-0., -1., -2., -3., -4.],
        [-1., -0., -1., -2., -3.],
        [-2., -1., -0., -1., -2.],
        [-3., -2., -1., -0., -1.],
        [-4., -3., -2., -1., -0.]])
attention without bias:
tensor([[0.2811, 0.1797, 0.1797, 0.1797, 0.1797],
        [0.1797, 0.2811, 0.1797, 0.1797, 0.1797],
        [0.1797, 0.1797, 0.2811, 0.1797, 0.1797],
        [0.1797, 0.1797, 0.1797, 0.2811, 0.1797],
        [0.1797, 0.1797, 0.1797, 0.1797, 0.2811]])
attention with relative bias:
tensor([[0.7324, 0.1723, 0.0634, 0.0233, 0.0086],
        [0.1481, 0.6294, 0.1481, 0.0545, 0.0200],
        [0.0527, 0.1431, 0.6085, 0.1431, 0.0527],
        [0.0200, 0.0545, 0.1481, 0.6294, 0.1481],
        [0.0086, 0.0233, 0.0634, 0.1723, 0.7324]])


## Rotary position embeddings


In [8]:
def rotate_half(x: torch.Tensor) -> torch.Tensor:
    x1 = x[..., ::2]
    x2 = x[..., 1::2]
    rotated = torch.stack((-x2, x1), dim=-1)
    return rotated.flatten(start_dim=-2)

def apply_rope(x: torch.Tensor, positions: torch.Tensor) -> torch.Tensor:
    dim = x.size(-1)
    inv_freq = 1.0 / (10000 ** (torch.arange(0, dim, 2).float() / dim))
    angles = positions.float().unsqueeze(-1) * inv_freq.unsqueeze(0)
    cos = torch.repeat_interleave(torch.cos(angles), 2, dim=-1)
    sin = torch.repeat_interleave(torch.sin(angles), 2, dim=-1)
    return x * cos + rotate_half(x) * sin

q = torch.tensor([[1.0, 0.0, 0.5, 0.5], [1.0, 0.0, 0.5, 0.5]])
k = torch.tensor([[1.0, 0.0, 0.5, 0.5], [1.0, 0.0, 0.5, 0.5]])
positions = torch.tensor([0, 3])

q_rot = apply_rope(q, positions)
k_rot = apply_rope(k, positions)

print('dot product at same relative offset:')
print(q_rot @ k_rot.T)


dot product at same relative offset:
tensor([[ 1.5000, -0.4902],
        [-0.4902,  1.5000]])


## External reference repos


In [9]:
references = {
    'rasbt': {
        'files': [
            'https://github.com/rasbt/LLMs-from-scratch/blob/main/ch04/01_main-chapter-code/gpt.py',
        ],
        'core': 'Rasbt keeps the teaching story simple with learned absolute position embeddings added to token embeddings before the blocks.',
    },
    'nanochat': {
        'files': [
            'https://github.com/karpathy/nanochat/blob/master/nanochat/gpt.py',
        ],
        'core': 'nanochat pushes position into attention itself through RoPE, which is closer to modern GPT-family practice.',
    },
}

for name, info in references.items():
    print(name.upper())
    print('core :', info['core'])
    print('files:')
    for file in info['files']:
        print(' -', file)
    print()


RASBT
core : Rasbt keeps the teaching story simple with learned absolute position embeddings added to token embeddings before the blocks.
files:
 - https://github.com/rasbt/LLMs-from-scratch/blob/main/ch04/01_main-chapter-code/gpt.py

NANOCHAT
core : nanochat pushes position into attention itself through RoPE, which is closer to modern GPT-family practice.
files:
 - https://github.com/karpathy/nanochat/blob/master/nanochat/gpt.py

